In [ ]:
# Used Claude Code to help convert between the full volume field noise and the noise realised at the electrodes on the surface

# Converting nominal noise to electrode-level noise

## Purpose

This notebook converts the full-volume noise percentage used by add_measurement_noise into the realised relative error at the electrode nodes. It covers all levels in the two measurement-noise sweeps, runs no inversions, and writes nothing to disk.

The generator uses

$$
\sigma_{\mathrm{noise}}=\eta_{\mathrm{vol}}\,\mathrm{RMS}_\Omega(\phi),
$$

whereas the measurements experience

$$
\eta_{\mathcal M}=
\frac{\lVert\mathrm{noise}\rVert_{\mathcal M}}
{\lVert\phi\rVert_{\mathcal M}}
\approx
\eta_{\mathrm{vol}}
\frac{\mathrm{RMS}_\Omega(\phi)}
{\mathrm{RMS}_{\mathcal M}(\phi)}.
$$

Because the surface field is weaker than the full-volume field, $\eta_{\mathcal{M}}$ is larger than the nominal percentage; the conversion factor for this experiment is about 1.87. *create-noise-figures.ipynb* recomputes the same conversion when generating the final-report figures.

In [ ]:
# One thread, set before torch is imported.
#
# The clean field solved below has to be the *same* clean field the two sweeps added
# their noise to, bit for bit: this notebook re-derives what those runs saw, and a
# field that differs in the last few bits gives a conversion factor for a slightly
# different experiment. Both sweeps solved it in the parent process under
# `torch.set_num_threads(1)`, and a threaded reduction sums in whatever order the
# threads finish in, so the pin is what makes this the same solve and not merely a
# similar one.
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import math
import platform
from pathlib import Path

import numpy as np
import scipy
import torch

import neuralphysics
from neuralphysics import (
    LIMB,
    SOURCE_SPECS,
    CoefficientHierarchy,
    ElectrodeArray,
    add_measurement_noise,
    configure,
    forward,
    gaussian_dipole,
    grid_at,
)

In [ ]:
print(configure())
torch.set_num_threads(1)

print(f"neuralphysics {neuralphysics.__version__} | torch {torch.__version__} | "
      f"numpy {np.__version__} | scipy {scipy.__version__} | python {platform.python_version()}")
print(f"default dtype {torch.get_default_dtype()} | torch threads {torch.get_num_threads()} | "
      f"{platform.platform()}")

## Forward settings

The grid, tissue model, sources, cycle count, and electrode mask match the measurement-noise sweep notebooks.

In [ ]:
N = 65          # nodes per axis; N = 2^k + 1 must hold for the v-cycle to work
NUM_CYCLES = 8  # fixed budget - not a convergence tolerance
WIDTH = 0.05    # the 1/e blob radius of a source, a physical length

# no RESULTS path: this notebook reads no saved run and writes no file. What it needs
# is the clean field, and that is solved below.

## Levels and draws

The notebook covers the shared noise-free control and nominal levels from \(0.25\%\) to \(4\%\). Reusing the original seed rule, \(2j+s\) for draw \(j\) and source \(s\), reproduces the noise fields used in the inversions.

In [ ]:
LEVELS_LOW = (0.25, 0.5, 0.75, 1.0)     # averaged-measurement-noise.ipynb
LEVELS_HIGH = (2.0, 3.0, 4.0)           # averaged-measurement-noise-2-4pc.ipynb
LEVELS = (0.0,) + LEVELS_LOW + LEVELS_HIGH

NOISE_SEEDS = (0, 1, 2, 3, 4)
CONTROL_SEEDS = (0,)


def draws_for(percent):
    """The noise draws run at `percent`, as both sweeps ran them."""
    return CONTROL_SEEDS if percent == 0.0 else NOISE_SEEDS


print(f"{len(LEVELS)} levels: " + ", ".join(f"{p:g}%" for p in LEVELS))
print("draws " + "/".join(str(len(draws_for(p))) for p in LEVELS))

## Clean fields

One noiseless field is solved for each source and reused for every conversion.

In [ ]:
grid = grid_at(N)

sources = [gaussian_dipole(grid, LIMB, width=WIDTH, r_blob_norm=r_norm, angle_deg=angle)
           for r_norm, angle in SOURCE_SPECS]

coeff_true = CoefficientHierarchy.from_vectors(LIMB, N)
phi_clean = [forward(I, coeff_true, num_cycles=NUM_CYCLES) for I in sources]

mask_nodes = ElectrodeArray().mask(N)   # full coverage of the four lateral faces

print(f"{len(phi_clean)} sources on a {N}^3 grid, "
      f"{int(mask_nodes.sum())} electrode nodes "
      f"({100 * ElectrodeArray().coverage(N):.0f}% of the lateral surface)\n")

print(f"{'source':>7} {'RMS_vol':>12} {'RMS_mask':>12} {'ratio':>8}")
for s, phi in enumerate(phi_clean):
    rms_vol = float(phi.pow(2).mean().sqrt())
    rms_mask = float(phi[mask_nodes].pow(2).mean().sqrt())
    print(f"{s:>7} {rms_vol:>12.4e} {rms_mask:>12.4e} {rms_vol / rms_mask:>8.3f}")

## Conversion

The sources are stacked to match the inverse objective:

\[
\eta_{\mathcal M}=
\sqrt{\frac{\sum_k\lVert\mathrm{noise}_k\rVert_{\mathcal M}^2}
{\sum_k\lVert\phi_k\rVert_{\mathcal M}^2}}.
\]

The expected relation is \(\eta_{\mathcal M}\approx K\eta_{\mathrm{vol}}\); the table reports the realised values, which vary slightly between draws. The noise-free control maps exactly to zero.

In [ ]:
def eta_measured(percent, draw):
    """Realised electrode-level relative error of one draw at one nominal level.

    The noise is redrawn rather than reconstructed: `add_measurement_noise` seeds a
    local generator, so the same (percent, seed) gives back the same field it gave
    the sweep, and the difference from the clean field is the noise that inversion
    actually saw.
    """
    if percent == 0.0:
        return 0.0                     # no draw is made; the control is the clean field

    noise_sq, signal_sq = 0.0, 0.0
    for s, phi in enumerate(phi_clean):
        phi_obs, _ = add_measurement_noise(phi, percent, len(phi_clean) * draw + s)
        noise_sq += float((phi_obs - phi)[mask_nodes].pow(2).sum())
        signal_sq += float(phi[mask_nodes].pow(2).sum())
    return math.sqrt(noise_sq / signal_sq)


# the level's figure is the mean over the draws made at it, which is the same
# aggregation every recovery table takes over its rows
ETA_M = {percent: float(np.mean([eta_measured(percent, draw)
                                 for draw in draws_for(percent)]))
         for percent in LEVELS}

K = math.sqrt(sum(float(phi.pow(2).mean()) for phi in phi_clean)
              / sum(float(phi[mask_nodes].pow(2).mean()) for phi in phi_clean))
print(f"K = {K:.4f}, so eta_M is about {K:.2f}x the nominal level\n")

In [ ]:
HEADERS_ETA = ("η_vol (%)", "η_M (%)")

rows_eta = [(f"{percent:.2f}", f"{100 * ETA_M[percent]:.2f}") for percent in LEVELS]

CAPTION_ETA = ("nominal noise against measured electrode-level noise\n"
               f"{len(NOISE_SEEDS)} draws averaged at each noisy level, "
               f"{len(CONTROL_SEEDS)} at the control; "
               "stacked over both sources on the electrode mask")

print(CAPTION_ETA + "\n")
print(f"{HEADERS_ETA[0]:>11}  {HEADERS_ETA[1]:>9}")
print("-" * 22)
for eta_vol, eta_m in rows_eta:
    print(f"{eta_vol:>11}  {eta_m:>9}")

## The nominal 5% case

This separate calculation reports both the mean over five draws and draw 0, which is the realisation used by *end-to-end-experiment.ipynb*. It is not added to the sweep results.

In [ ]:
# Standalone by design. 5% is not a level of either sweep, so it is not added to
# ETA_M - the tables index that dictionary, and keeping this out of it is what stops
# a 5% row appearing in a png that has no 5% runs behind it.
ETA_VOL_EXTRA = 5.0
assert ETA_VOL_EXTRA not in ETA_M, "keep the extra level out of the table's index"

per_draw_extra = [eta_measured(ETA_VOL_EXTRA, draw) for draw in NOISE_SEEDS]

print(f"η_vol = {ETA_VOL_EXTRA:.2f}%   (end-to-end-experiment.ipynb; not a sweep level)\n")
for draw, value in zip(NOISE_SEEDS, per_draw_extra):
    print(f"  draw {draw}   η_M = {100 * value:.2f}%")
print(f"\nmean over {len(NOISE_SEEDS)} draws   η_M = {100 * float(np.mean(per_draw_extra)):.2f}%")
print(f"draw 0 alone           η_M = {100 * per_draw_extra[0]:.2f}%")
print(f"analytic K x η_vol     η_M = {K * ETA_VOL_EXTRA:.2f}%")

## Result

For this geometry, electrode-level relative noise is approximately \(K=1.87\) times the nominal full-volume percentage. The table above gives the realised conversion at each level.